# ⚛️ Day 01a: ACID Properties — Atomicity, Consistency, Isolation, Durability

---

## 🎯 What You'll Master Today
- All four ACID properties with real examples
- Transaction lifecycle (BEGIN, COMMIT, ROLLBACK)
- WAL (Write-Ahead Logging) for durability
- ACID vs BASE (NoSQL tradeoff)

---

## 🏗️ The Bank Transfer Analogy

> Transferring $100 from Account A to B:
> **Atomic**: Both debit AND credit happen, or neither.
> **Consistent**: Total money before = total money after.
> **Isolated**: No one sees the "in-between" state (A debited, B not yet credited).
> **Durable**: Once confirmed, even a power failure can't undo it.

```
╔══════════════════════════════════════════════════════════════════╗
║                      ACID AT A GLANCE                            ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  A — Atomicity      │ All or nothing. No partial transactions.  ║
║  C — Consistency    │ DB moves from valid state to valid state. ║
║  I — Isolation      │ Concurrent txns don't interfere.          ║
║  D — Durability     │ Committed data survives crashes.          ║
║                                                                  ║
║  Enforced by:                                                    ║
║  A → Undo log (rollback on failure)                              ║
║  C → Constraints, triggers, application logic                    ║
║  I → Locks, MVCC, isolation levels                               ║
║  D → WAL (write-ahead log), fsync, replication                   ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. ATOMICITY — All or Nothing
# ============================================

class SimpleDB:
    """Simulates a database with transaction support."""
    def __init__(self):
        self.data = {}
        self.wal = []       # Write-Ahead Log
        self.snapshot = {}  # For rollback
        self.in_txn = False
    
    def begin(self):
        self.snapshot = dict(self.data)  # Save state
        self.in_txn = True
        self.wal.append("BEGIN")
        print("  [TXN] BEGIN")
    
    def set(self, key, value):
        old = self.data.get(key)
        self.data[key] = value
        self.wal.append(f"SET {key}: {old} → {value}")
        print(f"  [TXN] SET {key} = {value} (was {old})")
    
    def commit(self):
        self.wal.append("COMMIT")
        self.snapshot = {}
        self.in_txn = False
        print("  [TXN] COMMIT ✓")
    
    def rollback(self):
        self.data = dict(self.snapshot)
        self.wal.append("ROLLBACK")
        self.in_txn = False
        print("  [TXN] ROLLBACK — restored to snapshot")

# Demonstrate atomicity
db = SimpleDB()
db.data = {"account_A": 1000, "account_B": 500}
print(f"  Before: {db.data}  (total: {sum(db.data.values())})\n")

# Successful transfer
print("  === Successful Transfer ===")
db.begin()
db.set("account_A", db.data["account_A"] - 100)
db.set("account_B", db.data["account_B"] + 100)
db.commit()
print(f"  After: {db.data}  (total: {sum(db.data.values())})\n")

# Failed transfer (simulate error after debit)
print("  === Failed Transfer (error mid-transaction) ===")
db.begin()
db.set("account_A", db.data["account_A"] - 200)
print("  [ERROR] Connection lost! Cannot credit B!")
db.rollback()
print(f"  After rollback: {db.data}  (total: {sum(db.data.values())})")
print("\n  💡 Atomicity: debit was undone because credit failed. All or nothing!")

In [ ]:
# ============================================
# 2. CONSISTENCY — Valid State → Valid State
# ============================================

class ConsistentDB:
    def __init__(self):
        self.accounts = {"A": 1000, "B": 500}
        self.constraints = [
            lambda db: all(v >= 0 for v in db.accounts.values()),  # no negative balance
            lambda db: sum(db.accounts.values()) == 1500,          # total money conserved
        ]
    
    def check_constraints(self):
        for i, c in enumerate(self.constraints):
            if not c(self):
                return False, f"Constraint {i} violated"
        return True, "All constraints satisfied"
    
    def transfer(self, src, dst, amount):
        print(f"\n  Attempting: {src} → {dst}: ${amount}")
        # Simulate transaction
        old = dict(self.accounts)
        self.accounts[src] -= amount
        self.accounts[dst] += amount
        
        valid, msg = self.check_constraints()
        if valid:
            print(f"  ✓ {msg}. State: {self.accounts}")
        else:
            self.accounts = old  # rollback
            print(f"  ✗ {msg}. ROLLBACK! State: {self.accounts}")

cdb = ConsistentDB()
cdb.transfer("A", "B", 300)   # OK
cdb.transfer("A", "B", 800)   # Fails: A would go negative
print("\n  💡 Consistency: constraints checked before commit. Invalid → rollback!")

In [ ]:
# ============================================
# 3. ISOLATION — Concurrent txn interference
# ============================================

import threading, time

print("  === Without Isolation: Lost Update Problem ===")
shared = {"balance": 1000}
results_no_lock = []

def withdraw_no_lock(amount, name):
    balance = shared["balance"]       # READ
    # Simulate processing delay
    balance -= amount                  # COMPUTE
    shared["balance"] = balance        # WRITE
    results_no_lock.append(f"{name}: read 1000, wrote {balance}")

# Both read 1000, both subtract, last write wins
t1 = threading.Thread(target=withdraw_no_lock, args=(100, "T1"))
t2 = threading.Thread(target=withdraw_no_lock, args=(200, "T2"))
t1.start(); t2.start()
t1.join(); t2.join()

print(f"  Expected: 1000 - 100 - 200 = 700")
print(f"  Actual: {shared['balance']}")
for r in results_no_lock:
    print(f"    {r}")
print(f"  {'✗ LOST UPDATE!' if shared['balance'] != 700 else '✓ Correct'}")

print("\n  === With Isolation (Lock): No Lost Update ===")
shared = {"balance": 1000}
lock = threading.Lock()

def withdraw_with_lock(amount, name):
    with lock:
        balance = shared["balance"]
        balance -= amount
        shared["balance"] = balance
        print(f"    {name}: withdrew {amount}, balance now {balance}")

t1 = threading.Thread(target=withdraw_with_lock, args=(100, "T1"))
t2 = threading.Thread(target=withdraw_with_lock, args=(200, "T2"))
t1.start(); t2.start()
t1.join(); t2.join()
print(f"  Final: {shared['balance']}  {'✓' if shared['balance'] == 700 else '✗'}")

In [ ]:
# ============================================
# 4. DURABILITY — WAL (Write-Ahead Logging)
# ============================================

print("  How WAL ensures durability:\n")

class WALDatabase:
    def __init__(self):
        self.disk_data = {"balance": 1000}  # Simulates on-disk data
        self.wal_log = []                    # Write-Ahead Log (on disk FIRST)
        self.memory = {}                     # Buffer pool
    
    def transaction(self, operations):
        txn_id = len(self.wal_log) + 1
        
        # Step 1: Write to WAL FIRST (sequential, fast)
        print(f"  Step 1: Write to WAL (disk)")
        for op in operations:
            log_entry = f"TXN-{txn_id}: {op}"
            self.wal_log.append(log_entry)
            print(f"    WAL ← {log_entry}")
        self.wal_log.append(f"TXN-{txn_id}: COMMIT")
        print(f"    WAL ← TXN-{txn_id}: COMMIT")
        
        # Step 2: Apply to memory (fast)
        print(f"  Step 2: Apply to buffer pool (memory)")
        for op in operations:
            key, value = op.split("=")
            self.memory[key.strip()] = int(value.strip())
            print(f"    Memory: {key.strip()} = {value.strip()}")
        
        # Step 3: Eventually flush to disk (checkpoint)
        print(f"  Step 3: Checkpoint (flush memory → disk) [lazy]")
        self.disk_data.update(self.memory)
        print(f"    Disk: {self.disk_data}")
    
    def crash_recovery(self):
        print(f"\n  === CRASH RECOVERY ===")
        print(f"  Replay WAL entries after last checkpoint:")
        for entry in self.wal_log:
            print(f"    Replay: {entry}")
        print(f"  → Data restored to committed state!")

wal_db = WALDatabase()
wal_db.transaction(["balance = 900"])
wal_db.crash_recovery()

print("\n  💡 WAL: log written to disk BEFORE data pages. On crash → replay log.")

In [ ]:
# ============================================
# 5. ACID vs BASE comparison
# ============================================

print("""
  ┌─────────────────────┬───────────────────────────────────────────┐
  │ Property            │ ACID (SQL)          │ BASE (NoSQL)        │
  ├─────────────────────┼─────────────────────┼─────────────────────┤
  │ Focus               │ Correctness         │ Availability        │
  │ Consistency          │ Strong (immediate)  │ Eventual            │
  │ Availability during  │ May block/wait      │ Always available    │
  │ partition                                                       │
  │ Scaling focus        │ Vertical (bigger)   │ Horizontal (more)  │
  │ Use case             │ Banking, inventory  │ Social feeds, logs │
  └─────────────────────┴─────────────────────┴─────────────────────┘

  BASE = Basically Available, Soft state, Eventually consistent
  
  Transaction Lifecycle:
  ┌─────────────────────────────────────────────────────────────────┐
  │  BEGIN → READ/WRITE → {COMMIT | ROLLBACK}                     │
  │                                                                 │
  │  Active → Partially Committed → Committed                     │
  │      ↘                        ↗                                │
  │        Failed → Aborted (ROLLBACK)                             │
  └─────────────────────────────────────────────────────────────────┘
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Explain ACID with example? | Bank transfer: A=all-or-nothing, C=total preserved, I=no dirty reads, D=survives crash |
| 2 | How is Atomicity implemented? | Undo log — records old values for rollback |
| 3 | How is Durability implemented? | WAL (Write-Ahead Log) — log to disk before data pages |
| 4 | ACID vs BASE? | ACID: correctness-first (SQL). BASE: availability-first (NoSQL) |
| 5 | Can you have ACID in distributed systems? | Hard (2PC for A, consensus for C), often relax to eventual consistency |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Atomicity: undo log enables rollback on failure       │
## │  • Consistency: constraints + app logic guard validity   │
## │  • Isolation: locks/MVCC prevent concurrent corruption   │
## │  • Durability: WAL + checkpoint → crash recovery         │
## │  • BASE trades consistency for availability              │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Concurrency Control** — Locks, 2PL, MVCC, Timestamp Ordering